# GBDT vs NN

Thin orchestration notebook for the default `NN` vs `GBDT` analysis.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

In [2]:
CONFIG = "config_1.yaml"

In [3]:
from dataclasses import replace

from mfa import load_config

config_path = project_dir / "configs" / CONFIG
config = load_config(config_path)

# Read the configured value from YAML
yaml_n_jobs = config.parallelism.n_jobs

# Keep this notebook safe: allow only 1..2 workers for this run
run_n_jobs = min(max(yaml_n_jobs, 1), 2)
effective_n_jobs = 2

# Build a temporary config override for this run only (does not edit YAML)
run_config = replace(
    config,
    parallelism=replace(config.parallelism, n_jobs=effective_n_jobs),
)

if yaml_n_jobs != run_n_jobs:
    print(
        f"Loaded {config_path.name}; parallelism.n_jobs={yaml_n_jobs}. "
        f"Temporarily using n_jobs={run_n_jobs} for this run."
    )
else:
    print(
        f"Loaded {config_path.name}; parallelism.n_jobs={yaml_n_jobs}. "
        "Proceeding with the analysis."
    )

Loaded config_1.yaml; parallelism.n_jobs=-1. Temporarily using n_jobs=1 for this run.


In [4]:
from mfa import run_analysis

result = run_analysis(
    run_config,
    # datasets=[
    #     "Bank_Customer_Churn",
    #     "diabetes",
    #     "miami_housing",
    #     "physiochemical_protein",
    #     "website_phishing",
    # ],
)
result.analysis_table.head()

15:35:41 INFO mfa.pipeline: Starting analysis: comparisons=non_foundational_vs_foundational; scope=all benchmark datasets; unit=dataset; method_variant=default,tuned; n_jobs=2


15:35:41 INFO mfa.pipeline: Stage 1/5 raw results: cache hit (34110 rows, 51 dataset(s))
15:35:41 INFO mfa.pipeline: Stage 2/5 meta-features: trace enabled; metafeature caches remain active, so live per-split diagnostics appear only on cache misses
15:35:41 INFO mfa.pipeline: Stage 2/5 meta-features: pymfe enabled; rebuilding from split cache and reusing cached pymfe failures/incomplete outputs as-is
15:35:41 INFO mfa.pipeline: Stage 2/5 meta-features: building for all benchmark datasets
15:35:41 INFO mfa.metafeatures: Meta-features: preparing 51 dataset(s) with feature_sets=basic,redundancy,pymfe (n_jobs=2)
15:35:41 INFO mfa.metafeatures: Meta-features: trace enabled; split cache remains active, so live timing and warning diagnostics appear only on cache misses
15:35:41 INFO mfa.metafeatures: Meta-features: trace enabled with n_jobs=2; per-split logs may interleave. Use n_jobs=1 for ordered traces.
15:35:41 INFO mfa.metafeatures: Meta-features: submitting 816 split(s) across 51 datase

,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,...,best_a_error,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem
0,APSFailure,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,50666.666667,170.0,...,0.008650,0.832294,0.007284,0.333333,0.001366,0.002011,0.000670,0.498960,0.549148,0.183049
1,Amazon_employee_access,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,21846.000000,9.0,...,0.116781,0.001416,0.146592,0.778336,-0.029811,0.007122,0.002374,-0.776919,0.115570,0.038523
2,Another-Dataset-on-used-Fiat-500,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,30,1025.333333,7.0,...,731.881767,0.502696,719.852165,0.203675,12.029602,16.598990,3.030547,0.299021,0.414145,0.075612
3,Bank_Customer_Churn,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,6666.666667,10.0,...,0.129960,0.479097,0.125676,0.037007,0.004284,0.001790,0.000597,0.442090,0.159837,0.053279
4,Bioresponse,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,2500.666667,1776.0,...,0.126252,0.441583,0.123178,0.216194,0.003073,0.005005,0.001668,0.225389,0.360046,0.120015


In [5]:
from IPython.display import FileLink, display

split_keys = ["dataset", "repeat", "fold"]
fold_results = result.gap_table.merge(
    result.metafeature_table,
    on=split_keys,
    how="left",
    validate="many_to_one",
)

# `irregularity` is added at analysis-table time because it is z-scored across datasets.
if (
    "irregularity" in result.analysis_table.columns
    and "irregularity" not in fold_results.columns
):
    irregularity_lookup = result.analysis_table[
        ["dataset", "irregularity"]
    ].drop_duplicates(subset="dataset")
    fold_results = fold_results.merge(
        irregularity_lookup,
        on="dataset",
        how="left",
        validate="many_to_one",
    )

fold_results_csv = project_dir / "notebooks" / CONFIG.replace(".yaml", ".csv")
fold_results.to_csv(fold_results_csv, index=False)

print(f"Wrote fold-level results with meta-features to {fold_results_csv}")
print(f"Shape: {fold_results.shape}")
display(FileLink(str(fold_results_csv)))

Wrote fold-level results with meta-features to /work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/notebooks/config_1.csv
Shape: (816, 1131)


/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/notebooks/config_1.csv

In [6]:
import pandas as pd

# -- Inspect what the result object contains --
print(f"config_hash:        {result.config_hash}")
print(f"comparison_name:    {result.comparison_name}")
print(f"analysis_table:     {result.analysis_table.shape}")
print(f"gap_table:          {result.gap_table.shape}")
print(f"metafeature_table:  {result.metafeature_table.shape}")
print(f"correlation_results: {len(result.correlation_results)} features tested")
print(
    f"correction_result:  {result.correction_result.method if result.correction_result else None}"
)
print(f"multivariate_result: {result.multivariate_result}")

config_hash:        1bc820c4bb7b0f1f
comparison_name:    non_foundational_vs_foundational
analysis_table:     (51, 1132)
gap_table:          (816, 17)
metafeature_table:  (816, 1117)
correlation_results: 1114 features tested
correction_result:  bh
multivariate_result: None


## Correlation summary table

In [7]:
import numpy as np

# Build a comprehensive table from correlation + correction results
corr_df = pd.DataFrame([r.__dict__ for r in result.correlation_results])

if result.correction_result is not None:
    corr_df["p_value_adj"] = result.correction_result.adjusted_p_values
    corr_df["rejected"] = result.correction_result.rejected

# Add a significance star column for quick scanning
p_col = "p_value_adj" if "p_value_adj" in corr_df.columns else "p_value"
corr_df["sig"] = np.where(
    corr_df[p_col] < 0.001,
    "***",
    np.where(corr_df[p_col] < 0.01, "**", np.where(corr_df[p_col] < 0.05, "*", "")),
)

display_cols = [
    "predictor",
    "statistic",
    "ci_lower",
    "ci_upper",
    "p_value",
    *(["p_value_adj", "rejected"] if "p_value_adj" in corr_df.columns else []),
    "sig",
    "n_observations",
    "direction_confirmed",
]

corr_df[display_cols].sort_values("p_value")

,predictor,statistic,ci_lower,ci_upper,p_value,p_value_adj,rejected,sig,n_observations,direction_confirmed
548,pymfe__attr_ent.skewness,-0.510204,-0.708514,-0.252385,0.000154,0.121533,False,,50,None
792,pymfe__nodes_per_level.histogram.2,0.585154,0.289731,0.771920,0.000222,0.121533,False,,35,None
896,pymfe__tree_shape.histogram.6,0.561944,0.252978,0.765739,0.000443,0.161489,False,,35,None
626,pymfe__joint_ent.skewness,-0.489569,-0.727165,-0.178287,0.002103,0.196018,False,,37,None
517,pymfe__attr_conc.quantiles.1,0.420504,0.120506,0.667855,0.002362,0.196018,False,,50,None
...,...,...,...,...,...,...,...,...,...,...
964,pymfe__elite_nn.count,NaN,NaN,NaN,NaN,NaN,False,,33,None
1014,pymfe__naive_bayes.count,NaN,NaN,NaN,NaN,NaN,False,,33,None
1039,pymfe__one_nn.count,NaN,NaN,NaN,NaN,NaN,False,,33,None
1064,pymfe__random_node.count,NaN,NaN,NaN,NaN,NaN,False,,33,None


## Correlation scatter plots

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# sns.set_theme(style="ticks", context="talk")

# analysis = result.analysis_table.copy()
# predictors = corr_df["predictor"].tolist()
# n_preds = len(predictors)
# ncols = min(4, n_preds)
# nrows = int(np.ceil(n_preds / ncols))

# fig, axes = plt.subplots(
#     nrows, ncols, figsize=(5.5 * ncols, 5 * nrows), sharey=True, squeeze=False
# )

# for idx, predictor in enumerate(predictors):
#     ax = axes[idx // ncols][idx % ncols]
#     plot_df = analysis[[predictor, "delta_norm"]].dropna()

#     # Use log scale for features that are strictly positive and span orders of magnitude
#     use_log = (plot_df[predictor] > 0).all() and (
#         plot_df[predictor].max() / plot_df[predictor].min() > 10
#     )

#     sns.scatterplot(data=plot_df, x=predictor, y="delta_norm", s=35, alpha=0.85, ax=ax)
#     if use_log:
#         ax.set_xscale("log")

#     ax.set_xlabel(predictor)
#     if idx % ncols == 0:
#         ax.set_ylabel(r"$\Delta_{\ell\ell}$ (NN − GBDT)")
#     else:
#         ax.set_ylabel("")

#     # Annotate with correlation and adjusted p-value
#     row = corr_df.loc[corr_df["predictor"] == predictor].iloc[0]
#     p_display = row.get("p_value_adj", row["p_value"])
#     ax.text(
#         0.05,
#         0.95,
#         f"r = {row['statistic']:.3f}\np = {p_display:.3f}",
#         transform=ax.transAxes,
#         ha="left",
#         va="top",
#         fontsize=12,
#         bbox={"facecolor": "#d9d9d9", "edgecolor": "#9e9e9e", "alpha": 0.9},
#     )

# # Hide unused subplots
# for idx in range(n_preds, nrows * ncols):
#     axes[idx // ncols][idx % ncols].set_visible(False)

# sns.despine()
# fig.tight_layout()
# plt.show()